# Chapter 2: Pre-Model Workflow and Data Preprocessing

*Summary notebook for **scikit-learn Cookbook, Third Edition** — Chapter 2.*

> *"Garbage in, garbage out."* Data quality matters more than the choice of algorithm, and preprocessing is an **ongoing** task, not a one-off.

**Recipes covered**

1. The impact of raw data on model performance
2. Handling missing data (`SimpleImputer`, `KNNImputer`, `IterativeImputer`)
3. Scaling (`StandardScaler`, `MinMaxScaler`, `Normalizer`)
4. Encoding categorical variables (`OneHotEncoder`, `LabelEncoder`, `ColumnTransformer`)
5. Pipelines
6. Feature engineering (`PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel`)
7. Practical exercise: a comprehensive pipeline

In [1]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.precision', 3)

## 1. The Impact of Raw Data on Model Performance

### Summary
Models learn patterns from data; if the data is noisy, incomplete or biased, the model's ability to **generalise** drops. Common data issues and the scikit-learn tools that address them:

| Issue | Why it hurts | Tool |
|:--|:--|:--|
| Missing data | Most estimators refuse NaN | `SimpleImputer`, `KNNImputer`, `IterativeImputer` |
| Outliers | Distort means/variances | `RobustScaler`, z-score analysis |
| Categorical variables | Models need numbers | `OneHotEncoder`, `OrdinalEncoder` |
| Different scales | Distance/gradient methods dominated by large features | `StandardScaler`, `MinMaxScaler` |
| Data leakage | Over-optimistic metrics | Split first, then preprocess; use `Pipeline` |

Other helpers: `ColumnTransformer` (different preprocessing per column type) and `sklearn.feature_selection` (`SelectFromModel`, `RFE`).

### Theory explained — data leakage
**Leakage** occurs when information from outside the training set (e.g. test-set statistics) influences model building. If you compute the mean for scaling on the *whole* dataset before splitting, the test rows have already "voted" on the mean. Reported accuracy then overstates real-world performance. The cure is structural: **split first, fit preprocessing on train only**, which a `Pipeline` enforces automatically.

## 2. Handling Missing Data

### Setup — a toy dataset with ~20% missing values

In [2]:
np.random.seed(2024)
n_samples, n_features = 20, 10

data = {f"Feature{i+1}": np.random.uniform(0, 100, n_samples) for i in range(n_features)}
df = pd.DataFrame(data)

for column in df.columns:                       # knock out ~20% of entries
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

print("Missing values per column:")
print(df.isna().sum().to_dict())
df.head(6)

Missing values per column:
{'Feature1': 3, 'Feature2': 5, 'Feature3': 4, 'Feature4': 4, 'Feature5': 5, 'Feature6': 5, 'Feature7': 7, 'Feature8': 6, 'Feature9': 2, 'Feature10': 3}


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801,NaN,42.010,34.680,49.962,NaN,NaN,89.955,41.152,NaN
1,69.911,9.554,6.436,31.288,37.966,NaN,82.006,NaN,75.798,91.382
2,NaN,96.091,59.643,84.710,NaN,84.109,NaN,70.288,1.778,4.118
3,NaN,25.177,83.732,88.023,16.887,97.206,84.697,NaN,NaN,80.078
4,20.502,NaN,89.249,67.656,58.636,78.226,60.912,NaN,65.114,99.119
5,10.606,76.825,20.053,5.368,NaN,19.703,34.423,93.399,72.207,12.640


### 2.1 `SimpleImputer`
**Summary.** Replaces each NaN with a column statistic: `mean`, `median`, `most_frequent` or a `constant`.

**Theory.** For column $j$ with observed values $\mathcal{O}_j$: $\;\hat x_{ij}=\frac1{|\mathcal O_j|}\sum_{k\in\mathcal O_j}x_{kj}$ (mean strategy). It is fast and adds little bias when few values are missing, but it **ignores relationships between features** and shrinks the column's variance.

In [3]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
imputed_df = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
print("Learned means (first 3):", imputer.statistics_[:3].round(3))
imputed_df.head(6)

Learned means (first 3): [52.559 53.865 52.898]


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801,53.865,42.010,34.680,49.962,57.823,51.145,89.955,41.152,48.400
1,69.911,9.554,6.436,31.288,37.966,57.823,82.006,50.715,75.798,91.382
2,52.559,96.091,59.643,84.710,46.056,84.109,51.145,70.288,1.778,4.118
3,52.559,25.177,83.732,88.023,16.887,97.206,84.697,50.715,60.617,80.078
4,20.502,53.865,89.249,67.656,58.636,78.226,60.912,50.715,65.114,99.119
5,10.606,76.825,20.053,5.368,46.056,19.703,34.423,93.399,72.207,12.640


Every NaN in `Feature1` was replaced by the same value — that column's mean.

### 2.2 `KNNImputer`
**Summary.** Fills a missing value using the **k nearest rows** (measured on the features that are present).

**Theory.** For a row with a missing entry, find the $k$ most similar rows (Euclidean distance computed over co-observed features, via `nan_euclidean`) and take the **average** of their values for that feature. (The book describes a "majority label" — that is for categorical data; for numeric features the imputer *averages*.) It captures local structure but costs more and needs scaled features.

In [4]:
from sklearn.impute import KNNImputer

knn_imputer = KNNImputer(n_neighbors=2)
knn_imputed_df = pd.DataFrame(knn_imputer.fit_transform(df), columns=df.columns)
knn_imputed_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801,48.954,42.010,34.680,49.962,93.910,52.549,89.955,41.152,59.834
1,69.911,9.554,6.436,31.288,37.966,86.793,82.006,45.416,75.798,91.382
2,67.752,96.091,59.643,84.710,73.338,84.109,59.013,70.288,1.778,4.118
3,47.881,25.177,83.732,88.023,16.887,97.206,84.697,64.510,39.727,80.078
4,20.502,31.671,89.249,67.656,58.636,78.226,60.912,25.904,65.114,99.119
5,10.606,76.825,20.053,5.368,59.717,19.703,34.423,93.399,72.207,12.640


### 2.3 `IterativeImputer`
**Summary.** Models each feature that has missing values as a **regression on all the other features**, cycling round-robin until the estimates stabilise (similar in spirit to MICE). Still experimental, so it must be enabled explicitly.

**Theory.** Round $t$: for feature $j$, fit $x_j\sim f(x_{-j})$ on rows where $x_j$ is observed, predict the missing entries, then move to the next feature. Repeat until changes are small. It exploits multivariate relationships, so it can beat simple imputation — *but with purely random, uncorrelated toy data like ours there is nothing to learn*, so don't expect miracles here.

In [5]:
from sklearn.experimental import enable_iterative_imputer   # noqa: experimental feature
from sklearn.impute import IterativeImputer

iterative_imputer = IterativeImputer(random_state=2024)
iterative_imputed_df = pd.DataFrame(iterative_imputer.fit_transform(df), columns=df.columns)
iterative_imputed_df.head(6)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801,54.365,42.010,34.680,49.962,58.681,51.178,89.955,41.152,48.292
1,69.911,9.554,6.436,31.288,37.966,60.071,82.006,50.710,75.798,91.382
2,52.540,96.091,59.643,84.710,46.121,84.109,51.155,70.288,1.778,4.118
3,52.631,25.177,83.732,88.023,16.887,97.206,84.697,50.680,50.651,80.078
4,20.502,53.199,89.249,67.656,58.636,78.226,60.912,50.692,65.114,99.119
5,10.606,76.825,20.053,5.368,46.156,19.703,34.423,93.399,72.207,12.640


### Which imputer should I use? (the book's rule of thumb)

| Missing share | Suggestion |
|:--|:--|
| small (< ~5%) | `SimpleImputer` — light and low-bias *(the book's text says ">5%", which is a typo for "<5%")* |
| ~5–10% | `KNNImputer` (all-numeric) or `IterativeImputer` (mixed types) |
| > 10% | consider dropping the feature, or use domain-specific methods |

Also consider: data type, distribution, and the **missingness mechanism** (MCAR / MAR / MNAR). Tree models (decision tree, random forest, and `HistGradientBoosting`) can handle missing values natively, so imputation may be unnecessary for them.

## 3. Scaling Techniques

### Summary — standardization vs normalization
Terms are often confused:
- **Standardization** → mean 0, std 1 (`StandardScaler`).
- **Normalization (min–max)** → values in $[0,1]$ (`MinMaxScaler`).
- **Normalizer** → rescales each **row** (sample) to unit norm — a different thing again.

Scaling matters for **KNN, SVM, linear/logistic regression (gradient descent), PCA, neural nets**. It is unnecessary — even pointless — for **tree-based models**, which only use thresholds on raw feature order. (The book says scaling would be "detrimental" for trees; in practice it is simply *unnecessary*.)

### 3.1 `StandardScaler`
$$z=\frac{x-\mu}{\sigma}$$
Best when features are roughly Gaussian; outputs can be negative; sensitive to outliers (they inflate $\sigma$).

In [6]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Normalizer

scaled_df = pd.DataFrame(StandardScaler().fit_transform(iterative_imputed_df),
                         columns=iterative_imputed_df.columns)
print(scaled_df.agg(['mean', 'std']).round(3).iloc[:, :4])
scaled_df.head(4)

      Feature1  Feature2  Feature3  Feature4
mean    -0.000     0.000     0.000    -0.000
std      1.026     1.026     1.026     1.026


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,2.719e-01,0.019,-0.373,-0.894,0.189,0.032,1.905e-03,1.684e+00,-0.781,-0.004
1,7.560e-01,-1.875,-1.592,-1.044,-0.392,0.086,1.427e+00,-5.340e-04,0.652,1.429
2,-9.394e-04,1.784,0.231,1.312,0.003,1.022,8.540e-04,8.397e-01,-2.410,-1.472
3,3.022e-03,-1.214,1.057,1.458,-1.412,1.531,1.551e+00,-1.824e-03,-0.388,1.053


Means ≈ 0 and standard deviation ≈ 1 (pandas' `std` uses $n-1$, hence a value slightly above 1).

### 3.2 `MinMaxScaler`
$$x'=\frac{x-x_{\min}}{x_{\max}-x_{\min}}$$
Preserves the shape of the distribution and bounds values in $[0,1]$; very sensitive to outliers because they define min/max.

In [7]:
minmax_df = pd.DataFrame(MinMaxScaler().fit_transform(iterative_imputed_df),
                         columns=iterative_imputed_df.columns)
print(minmax_df.agg(['min', 'max']).iloc[:, :4])
minmax_df.head(4)

     Feature1  Feature2  Feature3  Feature4
min       0.0       0.0       0.0       0.0
max       1.0       1.0       1.0       1.0


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.604,0.518,3.805e-01,0.355,0.569,0.556,0.435,0.963,0.402,0.465
1,0.721,0.000,3.690e-04,0.314,0.413,0.572,0.833,0.539,0.756,0.919
2,0.537,1.000,5.690e-01,0.960,0.519,0.849,0.435,0.750,0.000,0.000
3,0.538,0.181,8.264e-01,1.000,0.139,1.000,0.868,0.538,0.499,0.800


### 3.3 `Normalizer` (row-wise)
$$\lVert\mathbf x\rVert_1=\sum_j|x_j|,\qquad \lVert\mathbf x\rVert_2=\sqrt{\sum_j x_j^2}$$
Each **sample** is divided by its norm so its length is 1 (default L2). Useful for sparse/text data or when only the *direction* of the vector matters (e.g. cosine similarity).

In [8]:
normalized_df = pd.DataFrame(Normalizer().fit_transform(iterative_imputed_df),
                             columns=iterative_imputed_df.columns)
print("Row L2 norms:", np.linalg.norm(normalized_df.values, axis=1).round(3)[:5])
normalized_df.head(4)

Row L2 norms: [1. 1. 1. 1. 1.]


,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.339,0.314,0.242,0.200,0.288,0.338,0.295,0.519,0.237,0.279
1,0.377,0.051,0.035,0.169,0.205,0.324,0.442,0.273,0.408,0.492
2,0.264,0.483,0.300,0.426,0.232,0.423,0.257,0.354,0.009,0.021
3,0.244,0.117,0.388,0.408,0.078,0.450,0.392,0.235,0.235,0.371


Every row now has L2 norm = 1 — contrast with the two column-wise scalers above.

**Algorithm sensitivity:** KNN (distance-based) and SVM are strongly affected by scale; linear regression converges faster with standardized features.

## 4. Encoding Categorical Variables

### Summary
Models need numbers. Variable types:
- **Nominal** – no order (colour, department) → one-hot encoding.
- **Ordinal** – ordered (Junior < Senior < Manager) → ordinal encoding.

In [9]:
np.random.seed(2024)
categorical_data = pd.DataFrame({
    "Department": np.random.choice(["A", "B", "C", "D"], size=20),
    "Position":   np.random.choice(["Junior", "Senior", "Manager"], size=20),
    "Location":   np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
})
categorical_data.head(6)

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA


### 4.1 `OneHotEncoder`
**Theory.** A category with $m$ levels becomes $m$ binary columns; exactly one is 1 per row. No artificial ordering is introduced. Downside: many features/levels → very wide, sparse matrices (the "curse of dimensionality"). Use `drop='first'` to avoid the redundant column in linear models, or `handle_unknown='ignore'` for unseen categories at prediction time.

In [10]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder(sparse_output=False)
onehot_encoded_df = pd.DataFrame(onehot_encoder.fit_transform(categorical_data),
                                 columns=onehot_encoder.get_feature_names_out())
print("Shape:", categorical_data.shape, "->", onehot_encoded_df.shape)
onehot_encoded_df.head(4)

Shape: (20, 3) -> (20, 11)


,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


### 4.2 `LabelEncoder`
**Theory.** Maps each category to an integer ($0,1,2,\dots$). Cheap, but models may read the integers as **ordered/distance-bearing**, which is wrong for nominal data. Important: `LabelEncoder` is designed for the **target** `y`; for **input features** scikit-learn recommends `OrdinalEncoder` (the book applies `LabelEncoder` to features for illustration only).

In [11]:
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

label_encoded_df = pd.DataFrame()
for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = LabelEncoder().fit_transform(categorical_data[column])
print(label_encoded_df.head(4))

# The recommended tool for features, with an explicit order for an ordinal variable:
ord_enc = OrdinalEncoder(categories=[["Junior", "Senior", "Manager"]])
print("\nOrdinalEncoder (Position):", ord_enc.fit_transform(categorical_data[["Position"]])[:6].ravel())

   Department_encoded  Position_encoded  Location_encoded
0                   0                 1                 1
1                   2                 1                 1
2                   0                 0                 2
3                   0                 1                 1

OrdinalEncoder (Position): [2. 2. 0. 2. 2. 1.]


### 4.3 `ColumnTransformer`
**Summary.** Applies *different* transformers to different columns of a mixed DataFrame and concatenates the results — the standard way to preprocess real tabular data.

In [12]:
from sklearn.compose import ColumnTransformer

np.random.seed(2024)
mixed_data = pd.DataFrame({
    "Age": np.random.randint(25, 65, size=20),
    "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
    "Experience": np.random.randint(1, 20, size=20),
    "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
    "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
})

column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="passthrough",
)
transformed = column_transformer.fit_transform(mixed_data)
if hasattr(transformed, "toarray"):
    transformed = transformed.toarray()

transformed_df = pd.DataFrame(transformed, columns=column_transformer.get_feature_names_out())
print("Shape:", mixed_data.shape, "->", transformed_df.shape)
transformed_df.head(4)

Shape: (20, 5) -> (20, 10)


,num__Age,num__Salary,num__Experience,cat__Department_Finance,cat__Department_HR,cat__Department_IT,cat__Department_Sales,cat__Position_Junior,cat__Position_Manager,cat__Position_Senior
0,-1.045,-0.305,0.327,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.728,1.105,1.262,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636,-1.582,1.075,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.358,-1.577,-0.608,0.0,0.0,1.0,0.0,1.0,0.0,0.0


## 5. Introduction to Pipelines

### Summary
A `Pipeline` is an ordered list of `(name, step)` pairs; all steps but the last must be transformers. Advantages: sequential execution, simpler code, consistency (no leakage), easy hyper-parameter tuning (`step__param`), modularity.

### Why split *before* transforming
1. **Avoids leakage** – statistics (means, std, imputation values) come from training data only.
2. **Realistic evaluation** – test data plays the role of unseen future data.
3. **Consistency** – the same fitted parameters transform train and test.

The book builds a demo pipeline on `transformed_df`, taking the last column as a stand-in "target" purely to demonstrate the mechanics.

In [13]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

X = transformed_df.iloc[:, :-1]     # all columns except last
y = transformed_df.iloc[:, -1]      # last column used as a dummy target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2024)

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

X_train_transformed = pd.DataFrame(pipeline.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_transformed  = pd.DataFrame(pipeline.transform(X_test),      columns=X_test.columns,  index=X_test.index)
print("Train:", X_train_transformed.shape, "Test:", X_test_transformed.shape)

Train: (16, 9) Test: (4, 9)


Note the asymmetry: `fit_transform` on train, only `transform` on test.

**Visualising a pipeline** — in a notebook, `set_config(display="diagram")` renders an interactive diagram.

In [14]:
from sklearn import set_config
set_config(display="diagram")
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['num__Age','num__Salary','num__Experience',...,'cat__Department_Sales', 'cat__Position_Junior','cat__Position_Manager']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'mean'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_

## 6. Feature Engineering

### Summary
Two activities: **feature extraction/creation** (derive new variables) and **feature selection** (keep only informative ones). Selection methods are *filter* (statistical scores), *wrapper* (evaluate subsets by training models) or *hybrid/embedded*.

### 6.1 `PolynomialFeatures`
**Theory.** Degree-2 expansion of $(a,b)$ gives $[1,\,a,\,b,\,a^2,\,ab,\,b^2]$: **powers** capture curvature, **cross-terms** capture interactions. With $p$ features and degree $d$ the output has $\binom{p+d}{d}$ columns — grows quickly.

In [15]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2)
poly_features = poly.fit_transform(X_train_transformed)
print("Features:", X_train_transformed.shape[1], "->", poly_features.shape[1])
pd.DataFrame(poly_features, columns=poly.get_feature_names_out(X_train_transformed.columns)).iloc[:3, :8]

Features: 9 -> 55


,1,num__Age,num__Salary,num__Experience,cat__Department_Finance,cat__Department_HR,cat__Department_IT,cat__Department_Sales
0,1.0,0.834,-0.867,-1.560,-0.577,-0.480,2.082,-0.775
1,1.0,-0.689,1.283,-0.151,-0.577,2.082,-0.480,-0.775
2,1.0,0.544,-1.173,-1.761,-0.577,-0.480,2.082,-0.775


### 6.2 `KBinsDiscretizer`
**Theory.** Turns a continuous feature into $k$ ordered bins. `strategy`:
- `uniform` – equal-width bins,
- `quantile` – equal number of samples per bin,
- `kmeans` – bin edges from 1-D K-Means clusters.
Useful to capture non-linear effects with linear models and to reduce outlier influence.

In [16]:
from sklearn.preprocessing import KBinsDiscretizer

kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")
binned_df = pd.DataFrame(kbins.fit_transform(X_train_transformed), columns=X_train_transformed.columns)
binned_df.head(5)

,num__Age,num__Salary,num__Experience,cat__Department_Finance,cat__Department_HR,cat__Department_IT,cat__Department_Sales,cat__Position_Junior,cat__Position_Manager
0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
1,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0
2,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0
3,0.0,2.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0
4,2.0,1.0,2.0,0.0,0.0,0.0,2.0,0.0,2.0


### 6.3 `RFE` — Recursive Feature Elimination (wrapper method)
**Theory.** Fit an estimator exposing importances (`coef_` or `feature_importances_`), drop the least important feature(s), refit, and repeat until `n_features_to_select` remain. `ranking_` gives 1 for the best, increasing for features eliminated earlier. `RFECV` adds cross-validation to choose the number of features.

In [17]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

rfe = RFE(estimator=LinearRegression(), n_features_to_select=1)
rfe.fit(X_train_transformed, y_train)
pd.Series(rfe.ranking_, index=X_train_transformed.columns, name="RFE rank").sort_values()

cat__Position_Manager      1
cat__Position_Junior       2
num__Age                   3
cat__Department_IT         4
cat__Department_HR         5
cat__Department_Finance    6
cat__Department_Sales      7
num__Experience            8
num__Salary                9
Name: RFE rank, dtype: int64

### 6.4 `SelectFromModel` (embedded method)
**Theory.** Fit a model, read its importance (`|coef_|` or `feature_importances_`), and keep features whose importance exceeds a **threshold** (here the mean importance). Works especially well with tree ensembles and L1-regularised linear models.

In [18]:
from sklearn.feature_selection import SelectFromModel

selector = SelectFromModel(estimator=LinearRegression(), prefit=False, threshold='mean')
selector.fit(X_train_transformed, y_train)

pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Importance': selector.estimator_.coef_,
    'Selected': selector.get_support(),
}).sort_values('Importance', key=abs, ascending=False)

,Feature,Importance,Selected
8,cat__Position_Manager,-5.000e-01,True
7,cat__Position_Junior,-4.330e-01,True
6,cat__Department_Sales,-9.116e-16,False
3,cat__Department_Finance,-8.701e-16,False
5,cat__Department_IT,-7.202e-16,False
4,cat__Department_HR,-5.140e-16,False
2,num__Experience,-1.996e-16,False
0,num__Age,-1.222e-16,False
1,num__Salary,0.000e+00,False


## 7. Practical Exercise — A Comprehensive Pipeline

**Task (from the book):** California Housing → split → pipeline with ≥3 steps ending in an estimator → fit → evaluate on test data.

> **Offline note.** `fetch_california_housing()` downloads data from the internet. If your environment is offline, the cell falls back to a **synthetic regression dataset** with missing values injected, so the exercise still runs end to end. On Colab/your laptop the real dataset loads.

In [19]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

try:
    from sklearn.datasets import fetch_california_housing
    housing = fetch_california_housing(as_frame=True)
    X_h, y_h = housing.data, housing.target
    source = "California Housing (real)"
except Exception:
    from sklearn.datasets import make_regression
    X_arr, y_h = make_regression(n_samples=2000, n_features=8, n_informative=5, noise=10, random_state=2024)
    X_h = pd.DataFrame(X_arr, columns=[f"f{i}" for i in range(8)])
    X_h = X_h.mask(np.random.RandomState(2024).rand(*X_h.shape) < 0.05)    # inject 5% NaN
    source = "synthetic fallback (offline)"
print("Data source:", source, "| shape:", X_h.shape)

X_tr, X_te, y_tr, y_te = train_test_split(X_h, y_h, test_size=0.2, random_state=2024)

full_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestRegressor(n_estimators=100, random_state=2024, n_jobs=-1)),
])
full_pipe.fit(X_tr, y_tr)
pred = full_pipe.predict(X_te)
print(f"Test R^2 : {r2_score(y_te, pred):.3f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_te, pred)):.3f}")

Data source: synthetic fallback (offline) | shape: (2000, 8)


Test R^2 : 0.882
Test RMSE: 50.351


Evaluation uses held-out data through the same pipeline: `r2_score` and RMSE ($\sqrt{\text{MSE}}$, in the units of the target). A random forest doesn't need scaling; it is kept here only to show a 3-step pipeline.

---
## Chapter 2 Summary

| Task | Tool | Remember |
|:--|:--|:--|
| Missing values | `SimpleImputer` / `KNNImputer` / `IterativeImputer` | Choose by % missing and feature relationships; trees can skip this |
| Scaling | `StandardScaler`, `MinMaxScaler`, `Normalizer` | Column-wise vs row-wise; not needed for trees |
| Categorical | `OneHotEncoder` (nominal), `OrdinalEncoder` (ordinal) | `LabelEncoder` is for targets |
| Mixed data | `ColumnTransformer` | Different steps per column group |
| Workflow | `Pipeline` | Split first; `fit_transform` train, `transform` test |
| New features | `PolynomialFeatures`, `KBinsDiscretizer` | Beware dimensional explosion |
| Selection | `RFE`, `SelectFromModel` | Wrapper vs embedded methods |